# Mortgage Approval Workflow with Oracle AI Agent Memory (OCI)

[![Documentation](https://img.shields.io/badge/Documentation-Oracle%20AI%20Agent%20Memory-red?style=flat-square)](https://www.oracle.com/database/ai-agent-memory/)

**Framework:** LangGraph `StateGraph` with OCI-backed stage reasoning · **Memory:** Oracle AI Agent Memory on Oracle AI Database

This notebook builds a **mortgage approval workflow**: a predetermined pipeline that runs every application through identity verification, credit assessment, income verification, debt-to-income calculation, and final decision. The workflow is deterministic; OCI Generative AI is used only to write concise stage rationales from deterministic tool results.

---

## Application Modes in Agent Applications

AI agent applications generally fall into three operational modes:

| Mode | Shape | Typical use |
|---|---|---|
| **Assistant** | Turn-by-turn conversational, tool-using | Operations support, customer service, internal copilots |
| **Deep Research** | Multi-step autonomous investigation | Literature review, scoping, market research |
| **Workflow** | Predetermined sequence of steps with conditional branches | Approval pipelines, compliance gates, structured triage |

> **This notebook focuses on the Workflow mode.**
>
> In Workflow mode, the shape of the work is fixed up-front. The graph controls routing and auditability; model calls stay local to stage-level explanation.


## What You'll Learn

- How to model a deterministic workflow as a **`StateGraph`** with typed state
- How to use LangChain's **prebuilt `create_agent`** as an intelligent node inside the graph
- How to drive conditional routing with **`add_conditional_edges`**
- How to use Oracle AI Agent Memory to persist applicant data and audit trails across the workflow's stages and across runs

> **💡 Key insight:** Memory is what lets a workflow survive a crash or a restart. When stage 3 fails mid-run, the state up to stage 3 is already in memory — the workflow can resume rather than restart. That is the practical difference between a memory-aware workflow and a script.

## Prerequisites

- **Oracle AI Database** running locally (Docker container) or reachable over the network
- **OCI Generative AI** access to `xai.grok-3-fast`
- Local OCI config file at `~/.oci/config`, or set `OCI_CONFIG_FILE` and `OCI_PROFILE`
- The `oracleagentmemory` wheel installed in the active kernel's environment


## 1. Install dependencies

In [ ]:
%pip install -q oracleagentmemory==26.4.0 langgraph oci nest_asyncio


## 2. Configure OCI and Oracle connection


In [ ]:
import configparser
import os
from pathlib import Path


def _expand_path(value: str) -> str:
    return os.path.expanduser(os.path.expandvars(value))


def _read_oci_profile_region(config_file: str, profile_name: str) -> str:
    config_path = Path(config_file).expanduser()
    if not config_path.exists():
        return ""

    parser = configparser.RawConfigParser()
    parser.read(config_path)
    if profile_name == "DEFAULT" and parser.defaults().get("region"):
        return parser.defaults()["region"].strip()
    for section in (profile_name, f"profile {profile_name}"):
        if parser.has_section(section) and parser.has_option(section, "region"):
            return parser.get(section, "region").strip()
    return ""


def _read_oci_profile_values(config_file: str, profile_name: str) -> dict[str, str]:
    config_path = Path(config_file).expanduser()
    if not config_path.exists():
        return {}

    parser = configparser.RawConfigParser()
    parser.read(config_path)
    if profile_name == "DEFAULT":
        values = dict(parser.defaults())
    else:
        values = {}
        for section in (profile_name, f"profile {profile_name}"):
            if parser.has_section(section):
                values.update(dict(parser.items(section)))
                break

    if values.get("key_file"):
        values["key_file"] = _expand_path(values["key_file"])
    return values


def _set_oci_env_aliases(config_file: str, profile_name: str) -> None:
    values = _read_oci_profile_values(config_file, profile_name)
    aliases = {
        "user": ("OCI_USER", "OCI_USER_ID"),
        "tenancy": ("OCI_TENANCY", "OCI_TENANCY_ID"),
        "fingerprint": ("OCI_FINGERPRINT",),
        "key_file": ("OCI_KEY_FILE",),
        "region": ("OCI_REGION", "OCI_REGION_NAME"),
    }
    for config_key, env_names in aliases.items():
        value = values.get(config_key, "")
        if not value:
            continue
        for env_name in env_names:
            os.environ.setdefault(env_name, value)


def _env(name: str, default: str = "") -> str:
    raw_value = os.environ.get(name)
    value = raw_value if raw_value not in (None, "") else default
    value = _expand_path(value) if name.endswith("_FILE") else value
    os.environ[name] = value
    return value


def _require_non_empty(value: str, name: str, hint: str) -> None:
    if not value:
        raise RuntimeError(f"Missing {name}. {hint}")


def _require_existing_file(value: str, name: str, hint: str) -> None:
    _require_non_empty(value, name, hint)
    if not Path(value).expanduser().exists():
        raise RuntimeError(f"{name} does not exist at {value}. {hint}")


LLM_PROVIDER = _env("LLM_PROVIDER", "oci_genai_native").strip().lower().replace("-", "_")
if LLM_PROVIDER not in {"oci_genai_native", "oci_native"}:
    raise RuntimeError("This OCI notebook supports LLM_PROVIDER=oci_genai_native only.")

LLM_MODEL = _env("LLM_MODEL", "xai.grok-3-fast").strip()
OCI_CONFIG_FILE = _env("OCI_CONFIG_FILE", "~/.oci/config")
OCI_PROFILE = _env("OCI_PROFILE", "DEFAULT")
OCI_PROFILE_REGION = _read_oci_profile_region(OCI_CONFIG_FILE, OCI_PROFILE)
OCI_GENAI_REGION = os.environ.get("OCI_GENAI_REGION", "").strip() or "us-chicago-1"
os.environ["OCI_GENAI_REGION"] = OCI_GENAI_REGION
OCI_GENAI_ENDPOINT = _env(
    "OCI_GENAI_ENDPOINT",
    f"https://inference.generativeai.{OCI_GENAI_REGION}.oci.oraclecloud.com",
)
OCI_PROFILE_VALUES = _read_oci_profile_values(OCI_CONFIG_FILE, OCI_PROFILE)
OCI_GENAI_COMPARTMENT_OCID = _env(
    "OCI_GENAI_COMPARTMENT_OCID",
    OCI_PROFILE_VALUES.get("compartment_id") or OCI_PROFILE_VALUES.get("tenancy", ""),
).strip()
OCI_GENAI_MODEL_OCID = _env("OCI_GENAI_MODEL_OCID", LLM_MODEL).strip()
OCI_GENAI_EMBED_MODEL = _env("OCI_GENAI_EMBED_MODEL", "cohere.embed-english-v3.0")

_set_oci_env_aliases(OCI_CONFIG_FILE, OCI_PROFILE)
os.environ.setdefault("OCI_REGION", OCI_GENAI_REGION)
os.environ.setdefault("OCI_REGION_NAME", OCI_GENAI_REGION)
os.environ.setdefault("OCI_COMPARTMENT_ID", OCI_GENAI_COMPARTMENT_OCID)

os.environ.setdefault("DB_USER", "VECTOR")
os.environ.setdefault("DB_PASSWORD", "VectorPwd_2025")
os.environ.setdefault("DB_CONNECT_STRING", "localhost:1521/FREEPDB1")

_require_existing_file(
    OCI_CONFIG_FILE,
    "OCI_CONFIG_FILE",
    "Set OCI_CONFIG_FILE to your OCI config path, usually ~/.oci/config.",
)
_require_non_empty(OCI_PROFILE, "OCI_PROFILE", "Set OCI_PROFILE to a profile in OCI_CONFIG_FILE.")
_require_non_empty(OCI_GENAI_REGION, "OCI_GENAI_REGION", "Set OCI_GENAI_REGION for OCI Generative AI.")
_require_non_empty(
    OCI_GENAI_COMPARTMENT_OCID,
    "OCI_GENAI_COMPARTMENT_OCID",
    "Set this to a compartment OCID with OCI Generative AI permissions. The notebook falls back to compartment_id or tenancy from OCI_CONFIG_FILE when present.",
)
_require_non_empty(
    OCI_GENAI_MODEL_OCID,
    "OCI_GENAI_MODEL_OCID",
    "Set this to the on-demand model OCID or model ID. Defaults to LLM_MODEL, for example xai.grok-3-fast.",
)

import nest_asyncio
nest_asyncio.apply()

print("Environment configured for OCI Generative AI.")
print(f"LLM provider: {LLM_PROVIDER}")
print(f"LLM model: {LLM_MODEL}")
print(f"OCI profile region: {OCI_PROFILE_REGION or 'not set'}")
print(f"OCI GenAI region: {OCI_GENAI_REGION}")
print(f"OCI endpoint: {OCI_GENAI_ENDPOINT}")
print(f"OCI compartment: {OCI_GENAI_COMPARTMENT_OCID[:18]}..." if OCI_GENAI_COMPARTMENT_OCID else "OCI compartment: not set")
print(f"OCI model id: {OCI_GENAI_MODEL_OCID}")
print(f"OCI embedding model: {OCI_GENAI_EMBED_MODEL}")


## 3. Connect to Oracle AI Database and initialise the memory client

The workflow uses memory for three things:

1. **Applicant records** — identity, financial data, third-party verifications — stored as durable memories keyed by applicant ID.
2. **Stage outcomes** — each node writes its decision and rationale, producing an auditable trail.
3. **Policy facts** — underwriting thresholds, approved document types — stored as `fact` records the agent nodes can recall.

> **💡 Key insight:** In a regulated workflow, memory is also your audit trail. Every stage's input, decision, and rationale should be persisted in a queryable substrate. Oracle AI Database gives you that substrate and the compliance controls already wrapped around it.

In [ ]:
import asyncio
import json
from collections.abc import Sequence
from typing import Any

import numpy as np
import oci
from oracleagentmemory.apis.embedders.embedder import IEmbedder
from oracleagentmemory.apis.llms.llm import ILlm, LlmResponse


class OciSdkEmbedder(IEmbedder):
    """OracleAgentMemory embedder backed directly by OCI Generative AI."""

    def __init__(self, client, compartment_id: str, model_id: str):
        self.client = client
        self.compartment_id = compartment_id
        self.model_id = model_id

    def embed(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        from oci.generative_ai_inference import models

        details = models.EmbedTextDetails()
        details.inputs = texts
        details.compartment_id = self.compartment_id
        details.serving_mode = models.OnDemandServingMode(model_id=self.model_id)
        details.truncate = models.EmbedTextDetails.TRUNCATE_END
        details.input_type = (
            models.EmbedTextDetails.INPUT_TYPE_SEARCH_QUERY
            if is_query
            else models.EmbedTextDetails.INPUT_TYPE_SEARCH_DOCUMENT
        )

        response = self.client.embed_text(details)
        embeddings = getattr(response.data, "embeddings", None)
        if not embeddings:
            raise RuntimeError("OCI embed_text returned no embeddings.")
        return np.asarray(embeddings, dtype=np.float32)

    async def embed_async(self, texts: list[str], *, is_query: bool = False) -> np.ndarray:
        return await asyncio.to_thread(self.embed, texts, is_query=is_query)


def create_oci_genai_client():
    try:
        oci_config = oci.config.from_file(OCI_CONFIG_FILE, OCI_PROFILE)
    except Exception as exc:
        raise RuntimeError(
            f"Could not load OCI config profile {OCI_PROFILE!r} from {OCI_CONFIG_FILE}: {exc}"
        ) from exc

    return oci.generative_ai_inference.GenerativeAiInferenceClient(
        config=oci_config,
        service_endpoint=OCI_GENAI_ENDPOINT,
        retry_strategy=oci.retry.NoneRetryStrategy(),
        timeout=(10, 240),
    )


def _message_content_to_text(content: Any) -> str:
    if content is None:
        return ""
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        parts = []
        for item in content:
            if isinstance(item, dict):
                parts.append(str(item.get("text") or item.get("content") or ""))
            else:
                parts.append(str(getattr(item, "text", "") or getattr(item, "content", "") or item))
        return "\n".join(part for part in parts if part)
    return str(content)


def _oci_text_content(models, text: Any):
    item = models.TextContent()
    if hasattr(models.TextContent, "TYPE_TEXT"):
        item.type = models.TextContent.TYPE_TEXT
    item.text = _message_content_to_text(text)
    return item


def _chat_message_to_oci(models, message: dict):
    role = (message.get("role") or "user").lower()
    content_items = [_oci_text_content(models, message.get("content"))]
    if role == "system":
        oci_message = models.SystemMessage()
        oci_message.role = models.SystemMessage.ROLE_SYSTEM
    elif role == "assistant":
        oci_message = models.AssistantMessage()
        oci_message.role = models.AssistantMessage.ROLE_ASSISTANT
    else:
        oci_message = models.UserMessage()
        oci_message.role = models.UserMessage.ROLE_USER
    oci_message.content = content_items
    return oci_message


def _extract_oci_text(message: Any) -> str:
    parts = []
    for item in getattr(message, "content", None) or []:
        text = getattr(item, "text", None)
        if text:
            parts.append(text)
    return "\n".join(parts)


def oci_chat_text(
    messages: list[dict[str, str]],
    *,
    temperature: float = 0.2,
    max_tokens: int = 1200,
    response_json_schema: dict[str, Any] | None = None,
) -> str:
    from oci.generative_ai_inference import models

    chat_request = models.GenericChatRequest()
    chat_request.api_format = models.BaseChatRequest.API_FORMAT_GENERIC
    chat_request.messages = [_chat_message_to_oci(models, message) for message in messages]
    chat_request.max_tokens = max_tokens
    chat_request.temperature = temperature
    chat_request.top_p = 1.0
    chat_request.top_k = 0
    chat_request.is_stream = False

    if response_json_schema:
        schema = models.ResponseJsonSchema()
        schema.name = "response"
        schema.schema = response_json_schema
        schema.is_strict = False
        response_format = models.JsonSchemaResponseFormat()
        response_format.type = models.JsonSchemaResponseFormat.TYPE_JSON_SCHEMA
        response_format.json_schema = schema
        chat_request.response_format = response_format

    serving_mode = models.OnDemandServingMode()
    serving_mode.model_id = OCI_GENAI_MODEL_OCID

    chat_detail = models.ChatDetails()
    chat_detail.serving_mode = serving_mode
    chat_detail.chat_request = chat_request
    chat_detail.compartment_id = OCI_GENAI_COMPARTMENT_OCID

    response = genai_client.chat(chat_detail)
    data = response.data
    chat_response = getattr(data, "chat_response", data)
    choices = getattr(chat_response, "choices", None) or []
    message = getattr(choices[0], "message", None) if choices else None
    return _extract_oci_text(message)


class OciSdkLlm(ILlm):
    """OracleAgentMemory LLM backed directly by OCI Generative AI."""

    def __init__(self, temperature: float = 0.2, max_tokens: int = 1200):
        self.temperature = temperature
        self.max_tokens = max_tokens

    def generate(
        self,
        prompt: str | Sequence[dict[str, str]],
        *,
        response_json_schema: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> LlmResponse:
        if isinstance(prompt, str):
            messages = [{"role": "user", "content": prompt}]
        else:
            messages = list(prompt)
        text = oci_chat_text(
            messages,
            temperature=kwargs.get("temperature", self.temperature),
            max_tokens=kwargs.get("max_tokens", self.max_tokens),
            response_json_schema=response_json_schema,
        )
        return LlmResponse(text=text)

    async def generate_async(
        self,
        prompt: str | Sequence[dict[str, str]],
        *,
        response_json_schema: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> LlmResponse:
        return await asyncio.to_thread(
            self.generate,
            prompt,
            response_json_schema=response_json_schema,
            **kwargs,
        )


genai_client = create_oci_genai_client()
oamp_embedder = OciSdkEmbedder(
    genai_client,
    compartment_id=OCI_GENAI_COMPARTMENT_OCID,
    model_id=OCI_GENAI_EMBED_MODEL,
)
oamp_llm = OciSdkLlm(temperature=0.2)
print("OCI Generative AI helpers ready.")

import oracledb
from oracleagentmemory.core import OracleAgentMemory

connection = oracledb.connect(
    user=os.environ["DB_USER"],
    password=os.environ["DB_PASSWORD"],
    dsn=os.environ["DB_CONNECT_STRING"],
)
print("Connected to Oracle AI Database:", connection.version)

memory_client = OracleAgentMemory(
    connection=connection,
    embedder=oamp_embedder,
    extract_memories=False,
    schema_policy="create_if_necessary",
    table_name_prefix="mortgage_",
)
print("Memory client ready.")


## 4. Register the underwriter (user) and the workflow agent

In a mortgage pipeline, the `user_id` typically represents the loan officer or underwriter who owns the file. The `agent_id` represents the workflow itself, so the audit trail is attributed to the pipeline version and owned by the right human.

In [ ]:
UNDERWRITER_ID = "underwriter-alex"
AGENT_ID = "mortgage-workflow-v1"

for create_fn, eid, info in [
    (memory_client.add_user, UNDERWRITER_ID, "Senior underwriter — residential mortgages."),
    (memory_client.add_agent, AGENT_ID, "Deterministic mortgage approval workflow (v1)."),
]:
    try:
        create_fn(eid, info)
        print(f"Registered: {eid}")
    except ValueError:
        print(f"Already exists: {eid}")

## 5. Seed underwriting policy as durable facts

Underwriting thresholds (minimum credit score, maximum DTI) don't live in code — they're policy. We store them as `fact` records so the agent nodes can recall them via search rather than hard-coding them, and so policy changes don't require code changes.

> **📌 Production pattern.** `fact` records are a first-class OAMP record type. They're indexed and searchable the same way memories are, but conceptually represent static reference data rather than learned information. Treat them as the memory layer's equivalent of configuration.

In [ ]:
store = memory_client._store   # fact/guideline convenience methods live on the store

POLICY_FACTS = [
    "Minimum acceptable credit score for conventional loans is 620.",
    "Maximum acceptable debt-to-income ratio is 43%.",
    "Accepted income documentation: W-2 forms, 1040 returns, pay stubs from the last 30 days.",
    "Identity verification requires a government-issued photo ID and a utility bill or bank statement.",
    "Loans above $1.5M require manual senior-underwriter review regardless of automated decision.",
]

fact_ids = store.add(
    contents=POLICY_FACTS,
    record_type="fact",
    user_ids=UNDERWRITER_ID,
    agent_ids=AGENT_ID,
)
print(f"Seeded {len(fact_ids)} policy facts.")

## 6. Define the workflow state

LangGraph uses a typed state dict (`TypedDict`) that each node mutates by returning partial updates. For a mortgage workflow, the state carries everything the pipeline accumulates — applicant data, verification outcomes, and the final decision.

> **💡 Key insight:** Typed state is what makes a LangGraph workflow debuggable. You can introspect any run by dumping the state; any audit question ("what was the DTI at decision time?") is answerable from state alone.

In [ ]:
from typing_extensions import TypedDict
from typing import Optional

class MortgageState(TypedDict, total=False):
    # Input
    applicant_id: str
    applicant_name: str
    loan_amount: float
    property_value: float
    # Stage outputs
    identity_verified: bool
    identity_notes: str
    credit_score: int
    credit_notes: str
    monthly_income: float
    monthly_debt: float
    income_notes: str
    dti_ratio: float
    # Decision
    decision: str        # "approve" | "deny" | "manual_review"
    decision_reason: str

print("MortgageState defined with", len(MortgageState.__annotations__), "fields.")

## 7. Helper — write a durable audit memory for each stage

Every node calls this helper so every stage's decision is persisted as a memory with consistent metadata. That gives us a complete audit trail per applicant, searchable semantically (for pattern analysis) and filterable by applicant ID (for file reviews).

In [ ]:
def log_stage(applicant_id: str, stage: str, outcome: str, rationale: str) -> str:
    """Write a structured audit memory for one workflow stage."""
    content = f"[{stage}] applicant={applicant_id} outcome={outcome}. {rationale}"
    return memory_client.add_memory(
        content,
        user_id=UNDERWRITER_ID,
        agent_id=AGENT_ID,
        metadata={"applicant_id": applicant_id, "stage": stage, "outcome": outcome},
    )

print("log_stage() ready.")

## 8. Define stage tools

In a real pipeline each stage calls external services such as credit bureaus, document verification APIs, and income validators. For this notebook we stub them so the workflow runs end-to-end.

> **The tools are deterministic stubs.** In production, swap the fixtures for real API calls and keep the graph shape unchanged.


In [ ]:
IDENTITY_FIXTURES = {
    "A-001": {"verified": True, "notes": "Driver's license + utility bill matched."},
    "A-002": {"verified": True, "notes": "Passport + bank statement matched."},
    "A-003": {"verified": False, "notes": "Utility bill address mismatch; requires re-submission."},
}
CREDIT_FIXTURES = {
    "A-001": {"score": 742, "notes": "No delinquencies in past 24 months. Experian."},
    "A-002": {"score": 598, "notes": "One 60-day delinquency in past 12 months. TransUnion."},
    "A-003": {"score": 710, "notes": "Thin file but no negatives. Equifax."},
}
INCOME_FIXTURES = {
    "A-001": {"monthly_income": 11_500, "monthly_debt": 2_100, "notes": "2024 W-2 + 30 days pay stubs."},
    "A-002": {"monthly_income": 6_800, "monthly_debt": 3_400, "notes": "1040 return only; no pay stubs."},
    "A-003": {"monthly_income": 9_200, "monthly_debt": 1_600, "notes": "W-2 + pay stubs verified."},
}


def check_identity_documents(applicant_id: str) -> dict:
    """Verify identity documents for an applicant. Returns pass/fail + notes."""
    return IDENTITY_FIXTURES.get(applicant_id, {"verified": False, "notes": "No documents on file."})


def pull_credit_report(applicant_id: str) -> dict:
    """Pull an applicant's credit score and summary. Returns score + bureau notes."""
    return CREDIT_FIXTURES.get(applicant_id, {"score": 0, "notes": "No credit file."})


def verify_income_documents(applicant_id: str) -> dict:
    """Verify income from supporting documents. Returns monthly income + monthly debt."""
    return INCOME_FIXTURES.get(applicant_id, {"monthly_income": 0, "monthly_debt": 0, "notes": "No income docs."})


print("Stage tools defined: check_identity_documents, pull_credit_report, verify_income_documents")


## 9. Build the workflow nodes

Each node is a plain Python function that takes the current state, calls deterministic stage tools, optionally asks OCI Generative AI for a concise rationale, writes an audit memory, and returns a partial state update. LangGraph merges the return value into the state automatically.

> **Key insight:** The policy gates stay deterministic. OCI is used to turn structured results into readable, auditable rationale text, not to decide whether a gate passes.


In [ ]:
import json


def _stage_rationale(stage: str, instruction: str, payload: dict) -> str:
    return oci_chat_text(
        [
            {"role": "system", "content": "You write concise mortgage underwriting audit notes."},
            {
                "role": "user",
                "content": (
                    f"Stage: {stage}\nInstruction: {instruction}\n"
                    f"Structured result:\n{json.dumps(payload, indent=2)}\n\n"
                    "Return only the requested audit note format."
                ),
            },
        ],
        temperature=0.1,
        max_tokens=400,
    ).strip()


def identity_node(state: MortgageState) -> dict:
    result = check_identity_documents(state["applicant_id"])
    verified = bool(result.get("verified"))
    text = _stage_rationale(
        "identity",
        "Return two lines: 'verified: true' or 'verified: false', then one sentence of notes.",
        result,
    )
    log_stage(state["applicant_id"], "identity", "pass" if verified else "fail", text)
    return {"identity_verified": verified, "identity_notes": text}


def credit_node(state: MortgageState) -> dict:
    result = pull_credit_report(state["applicant_id"])
    score = int(result.get("score") or 0)
    text = _stage_rationale(
        "credit",
        "Return two lines: 'score: <int>' and one sentence on notable credit items.",
        result,
    )
    log_stage(state["applicant_id"], "credit", f"score={score}", text)
    return {"credit_score": score, "credit_notes": text}


def income_node(state: MortgageState) -> dict:
    result = verify_income_documents(state["applicant_id"])
    inc = float(result.get("monthly_income") or 0.0)
    dbt = float(result.get("monthly_debt") or 0.0)
    text = _stage_rationale(
        "income",
        "Return three lines: 'monthly_income: <float>', 'monthly_debt: <float>', and one sentence of notes.",
        result,
    )
    log_stage(state["applicant_id"], "income", f"inc={inc},debt={dbt}", text)
    return {"monthly_income": inc, "monthly_debt": dbt, "income_notes": text}


def dti_node(state: MortgageState) -> dict:
    inc = state.get("monthly_income", 0.0) or 0.0
    dbt = state.get("monthly_debt", 0.0) or 0.0
    dti = (dbt / inc) if inc > 0 else 1.0
    log_stage(
        state["applicant_id"],
        "dti",
        f"ratio={dti:.3f}",
        f"Computed DTI from monthly_income={inc:.2f}, monthly_debt={dbt:.2f}.",
    )
    return {"dti_ratio": dti}


def decide_node(state: MortgageState) -> dict:
    score = state.get("credit_score", 0) or 0
    dti = state.get("dti_ratio", 1.0) or 1.0
    ident = state.get("identity_verified", False)
    loan = state.get("loan_amount", 0.0) or 0.0

    if not ident:
        decision, reason = "deny", "Identity not verified."
    elif score < 620:
        decision, reason = "deny", f"Credit score {score} below 620 minimum."
    elif dti > 0.43:
        decision, reason = "deny", f"DTI {dti:.2%} exceeds 43% ceiling."
    elif loan > 1_500_000:
        decision, reason = "manual_review", f"Loan ${loan:,.0f} above $1.5M jumbo threshold."
    else:
        decision, reason = "approve", f"All gates passed (score {score}, DTI {dti:.2%})."
    log_stage(state["applicant_id"], "decision", decision, reason)
    return {"decision": decision, "decision_reason": reason}


print("Nodes defined: identity_node, credit_node, income_node, dti_node, decide_node")


## 10. Assemble the workflow as a `StateGraph`

Nodes become vertices; `add_edge` draws the mandatory arrows; `add_conditional_edges` draws the routed ones. A conditional edge is how we short-circuit the pipeline when identity verification fails — no point pulling credit for an applicant whose identity is unverified.

> **💡 Key insight:** Conditional edges are what turn a linear script into a real workflow. They let you encode the policy decisions ("if identity fails, skip to denial") in the graph topology instead of spreading if-statements across node bodies.

In [ ]:
from langgraph.graph import StateGraph, START, END

def route_after_identity(state: MortgageState) -> str:
    return "credit" if state.get("identity_verified") else "decide"

def route_after_credit(state: MortgageState) -> str:
    return "income" if (state.get("credit_score") or 0) >= 620 else "decide"

builder = StateGraph(MortgageState)
builder.add_node("identity", identity_node)
builder.add_node("credit", credit_node)
builder.add_node("income", income_node)
builder.add_node("dti", dti_node)
builder.add_node("decide", decide_node)

builder.add_edge(START, "identity")
builder.add_conditional_edges("identity", route_after_identity,
                              {"credit": "credit", "decide": "decide"})
builder.add_conditional_edges("credit", route_after_credit,
                              {"income": "income", "decide": "decide"})
builder.add_edge("income", "dti")
builder.add_edge("dti", "decide")
builder.add_edge("decide", END)

graph = builder.compile()
print("Workflow compiled.")

## 11. Run the workflow end-to-end

We run three applicants through the pipeline. Each produces a full audit trail in Oracle AI Agent Memory and ends with an explicit decision.

In [ ]:
applicants = [
    {"applicant_id": "A-001", "applicant_name": "J. Patel",    "loan_amount":   480_000, "property_value":   600_000},
    {"applicant_id": "A-002", "applicant_name": "M. Johansson", "loan_amount":   320_000, "property_value":   400_000},
    {"applicant_id": "A-003", "applicant_name": "R. Okafor",   "loan_amount": 1_800_000, "property_value": 2_100_000},
]

for a in applicants:
    print(f"\n{'=' * 70}\nRunning workflow for {a['applicant_id']} ({a['applicant_name']})\n{'=' * 70}")
    final_state = graph.invoke(a)
    print(f"Decision: {final_state['decision'].upper()}")
    print(f"Reason:   {final_state['decision_reason']}")
    if final_state.get("credit_score"):
        print(f"  credit_score: {final_state['credit_score']}")
    if final_state.get("dti_ratio"):
        print(f"  dti: {final_state['dti_ratio']:.2%}")

## 12. Stream a workflow run for debugging

`graph.stream(..., stream_mode="updates")` yields a dict per node as it executes. Great for notebooks and for surfacing stage-by-stage progress in an operator UI.

In [ ]:
print("Streaming workflow for a new applicant:")
new_app = {
    "applicant_id": "A-004", "applicant_name": "D. Nguyen",
    "loan_amount": 525_000, "property_value": 650_000,
}

IDENTITY_FIXTURES["A-004"] = {"verified": True, "notes": "Passport + utility bill match."}
CREDIT_FIXTURES["A-004"] = {"score": 705, "notes": "Clean file, thin but positive."}
INCOME_FIXTURES["A-004"] = {"monthly_income": 10_200, "monthly_debt": 2_900, "notes": "W-2 + pay stubs verified."}

for chunk in graph.stream(new_app, stream_mode="updates"):
    for node_name, update in chunk.items():
        print(f"  [{node_name}] -> {list(update.keys())}")


## 13. Inspect the audit trail for one applicant

Every stage wrote a structured memory with `applicant_id` metadata. Pulling the trail is one memory search away.

In [ ]:
# Get the complete stage trail for A-001 via metadata filter
trail = store.list(
    "memory",
    user_id=UNDERWRITER_ID,
    agent_id=AGENT_ID,
    metadata_filter={"applicant_id": "A-001"},
    limit=50,
)
print(f"Audit trail for A-001 ({len(trail)} entries):")
for r in trail:
    stage = (r.metadata or {}).get("stage", "?")
    outcome = (r.metadata or {}).get("outcome", "?")
    print(f"  [{stage:<8}] [{outcome}] {r.content[:100]}")

## 14. Recall a policy fact

Because we stored underwriting thresholds as `fact` records, any node — or any downstream reporting tool — can recall them with semantic search. Policy changes become a memory update rather than a code change.

In [ ]:
policy_hits = memory_client.search(
    "maximum debt to income ratio",
    user_id=UNDERWRITER_ID, agent_id=AGENT_ID,
    record_types=["fact"], max_results=3,
)
print("Relevant policy facts:")
for r in policy_hits:
    print(f"  - {r.content}  [distance={r.distance:.3f}]")

## 15. Cleanup

In [ ]:
connection.close()
print("Connection closed.")

## Key Takeaways

> **1. Workflow mode is deterministic orchestration over local model reasoning.** The graph topology is fixed; OCI only writes stage-level rationale from structured results.

> **2. LangGraph keeps the workflow auditable.** Node boundaries, conditional routes, and final decisions are explicit in code.

> **3. Memory is your audit trail.** Every stage's decision and rationale is persisted with structured metadata (`applicant_id`, `stage`, `outcome`).

> **4. Policy belongs in memory, not in code.** Underwriting thresholds, accepted document types, and compliance rules can be stored as durable facts and recalled via search.

> **5. Oracle AI Database is the right substrate for regulated workflows.** The audit trail, applicant state, policy facts, and conversation transcripts all live in one governed backend.
